<div align="center">

# Redes Informadas por la Física (PINNs)

### Extensión a ecuaciones en derivadas parciales

**Lenin Herrera Padilla** · lenin.herrera.p@uni.pe

</div>

---

Este notebook muestra que el mismo esquema de las PINNs para EDO se extiende a ecuaciones en derivadas parciales: la red recibe más entradas, $(x,t)$ o $(x_1,x_2,t)$, y el residuo usa derivadas parciales calculadas por diferenciación automática.

| Caso | Ecuación | Tipo |
|---|---|---|
| 1 | Calor 1D: $u_t=\alpha\,u_{xx}$ | directo, solución exacta |
| 2 | Calor 1D: estimación de $\alpha$ | inverso, datos con ruido |
| 3 | Onda 1D: cuerda vibrante $u_{tt}=c^2u_{xx}$ | directo, solución exacta |
| 4 | Onda 2D con velocidad variable $c(x)$ | directo, referencia por diferencias finitas |

El caso 4 es una versión reducida del problema de propagación de ondas sísmicas de Moseley (2022).

**Entorno:** Google Colab · Python 3 · PyTorch, DeepXDE, NumPy, Matplotlib. El caso 4 tarda bastante en CPU; en Colab conviene activar la GPU (*Entorno de ejecución → Cambiar tipo de entorno*).

> Con `RAPIDO = True` los entrenamientos se reducen unas diez veces, para una demostración rápida.

## 0. Configuración del entorno

In [ ]:
RAPIDO = False        # True: entrenamientos cortos para una demostración en vivo

def IT(n):
    return max(n // 10, 300) if RAPIDO else n

import importlib.util, subprocess, sys
if importlib.util.find_spec("deepxde") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deepxde"], check=True)

import os
os.environ["DDE_BACKEND"] = "pytorch"
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import deepxde as dde

LBFGS_MAX = 1000 if RAPIDO else 15000
dde.optimizers.config.set_LBFGS_options(maxiter=LBFGS_MAX)

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
AZUL, ROJO, VERDE, GRIS = "#1F5AA6", "#C0392B", "#2E8B57", "#7F7F7F"

def error_l2(aprox, exacta):
    aprox, exacta = np.ravel(aprox), np.ravel(exacta)
    return np.linalg.norm(aprox - exacta) / np.linalg.norm(exacta)

def mapas(XX, TT, ue, up, ejes=("x", "t"), titulo=""):
    # solución de referencia, PINN y error absoluto
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for a, Z, nombre, cm in [(ax[0], ue, "(a) Referencia", "viridis"), (ax[1], up, "(b) PINN", "viridis"),
                             (ax[2], np.abs(up - ue), "(c) Error absoluto", "magma")]:
        im = a.pcolormesh(XX, TT, Z, cmap=cm, shading="auto"); fig.colorbar(im, ax=a)
        a.set_title(nombre); a.set_xlabel(ejes[0]); a.set_ylabel(ejes[1])
    if titulo: fig.suptitle(titulo)
    fig.tight_layout(); plt.show()

print("PyTorch", torch.__version__, "| DeepXDE", dde.__version__, "| modo rápido:", RAPIDO)

## De EDO a EDP

| | EDO | EDP |
|---|---|---|
| Red | $u_\theta(t)$ | $u_\theta(x,t)$ |
| Residuo | $R=u_\theta'-f(t,u_\theta)$ | $R=\dfrac{\partial u_\theta}{\partial t}-\mathcal{N}[u_\theta]$ |
| Condiciones | inicial | inicial y de frontera |
| Derivadas | diferenciación automática | diferenciación automática (parciales) |

$$\mathcal{L}(\theta)=\frac1{N_c}\sum_{i}R(x_i,t_i)^2+\frac1{N_0}\sum_j\big(u_\theta(x_j,0)-u_0(x_j)\big)^2+\frac1{N_b}\sum_k\big(u_\theta(x_k,t_k)-g(x_k,t_k)\big)^2$$

## 1. Ecuación del calor (problema directo)

$$u_t=\alpha\,u_{xx},\quad (x,t)\in[0,1]^2,\quad \alpha=0{,}1,\qquad u(x,0)=\mathrm{sen}(\pi x),\quad u(0,t)=u(1,t)=0.$$

**Solución analítica por separación de variables.** Con $u=X(x)\,T(t)$:
$$\frac{T'}{\alpha T}=\frac{X''}{X}=-\lambda\;\Rightarrow\;X=\mathrm{sen}(n\pi x),\quad \lambda=(n\pi)^2,\quad T=e^{-\alpha n^2\pi^2 t}.$$
La condición inicial selecciona $n=1$: $\;u(x,t)=e^{-\alpha\pi^2t}\,\mathrm{sen}(\pi x)$.

**PINN:** $R=\partial_t u_\theta-\alpha\,\partial_{xx}u_\theta$.

**Arquitectura:** $2-32-32-32-1$ · $\tanh$ · Adam $10\,000$ + L-BFGS · 2000 puntos interiores, 100 de frontera y 100 iniciales.

In [ ]:
ALPHA = 0.1
geom_xt = dde.geometry.GeometryXTime(dde.geometry.Interval(0, 1), dde.geometry.TimeDomain(0, 1))
exacta_calor = lambda X: np.exp(-ALPHA * np.pi ** 2 * X[:, 1:2]) * np.sin(np.pi * X[:, 0:1])
bc_cero = dde.icbc.DirichletBC(geom_xt, lambda X: 0 * X[:, 0:1], lambda X, on: on)           # u(0,t)=u(1,t)=0
ic_seno = dde.icbc.IC(geom_xt, lambda X: np.sin(np.pi * X[:, 0:1]), lambda _, on: on)        # u(x,0)=sen(pi x)

def calor(X, u):
    # X[:, 0] = x, X[:, 1] = t
    return dde.grad.jacobian(u, X, i=0, j=1) - ALPHA * dde.grad.hessian(u, X, i=0, j=0)

dde.config.set_random_seed(1234)
data = dde.data.TimePDE(geom_xt, calor, [bc_cero, ic_seno], num_domain=2000, num_boundary=100,
                        num_initial=100, num_test=2000)
modelo_calor = dde.Model(data, dde.nn.FNN([2] + [32] * 3 + [1], "tanh", "Glorot uniform"))
t0 = time.perf_counter()
modelo_calor.compile("adam", lr=1e-3); modelo_calor.train(iterations=IT(10000), display_every=2500)
modelo_calor.compile("L-BFGS"); modelo_calor.train()
tiempo = time.perf_counter() - t0

xs = np.linspace(0, 1, 201); XX, TT = np.meshgrid(xs, xs)
Xg = np.c_[XX.ravel(), TT.ravel()]
ue = exacta_calor(Xg).reshape(XX.shape); up = modelo_calor.predict(Xg).reshape(XX.shape)
print(f"\nCalor (directo): error L2 relativo = {error_l2(up, ue):.2e}   tiempo = {tiempo:.0f} s")
mapas(XX, TT, ue, up)

## 2. Ecuación del calor (problema inverso): estimación de $\alpha$

Se desconoce la difusividad $\alpha$ y se dispone de **60 mediciones** de $u(x,t)$ en puntos aleatorios, con 1 % de ruido. $\alpha$ se declara como variable entrenable, iniciada en $0{,}5$ (el valor real es $0{,}1$):

$$\mathcal{L}(\theta,\alpha)=\frac1{N_c}\sum R^2+\frac1{N_0}\sum\big(u_\theta(x_j,0)-\mathrm{sen}\,\pi x_j\big)^2+\frac1{N_b}\sum u_\theta^2+\frac1{60}\sum_{k=1}^{60}\big(u_\theta(x_k,t_k)-u_k\big)^2.$$

**Arquitectura:** $2-32-32-32-1$ · $\tanh$ · Adam $15\,000$ + L-BFGS.

In [ ]:
rng = np.random.default_rng(0)
X_obs = np.c_[rng.uniform(0, 1, 60), rng.uniform(0, 1, 60)]
u_obs = exacta_calor(X_obs) * (1 + 0.01 * rng.standard_normal((60, 1)))      # 1 % de ruido

dde.config.set_random_seed(1234)
alpha_var = dde.Variable(0.5)                                                  # valor real: 0,1
def calor_inv(X, u):
    return dde.grad.jacobian(u, X, i=0, j=1) - alpha_var * dde.grad.hessian(u, X, i=0, j=0)

obs = dde.icbc.PointSetBC(X_obs, u_obs)
data = dde.data.TimePDE(geom_xt, calor_inv, [bc_cero, ic_seno, obs], num_domain=2000, num_boundary=100,
                        num_initial=100, anchors=X_obs)
modelo_inv = dde.Model(data, dde.nn.FNN([2] + [32] * 3 + [1], "tanh", "Glorot uniform"))
seg = dde.callbacks.VariableValue(alpha_var, period=max(IT(15000) // 30, 1), filename="alpha.dat")
modelo_inv.compile("adam", lr=1e-3, external_trainable_variables=[alpha_var])
modelo_inv.train(iterations=IT(15000), display_every=5000, callbacks=[seg])
modelo_inv.compile("L-BFGS", external_trainable_variables=[alpha_var])
modelo_inv.train(callbacks=[seg])

a_est = float(alpha_var.detach())
print(f"\nα estimado = {a_est:.5f}   real = {ALPHA}   error = {100 * abs(a_est - ALPHA) / ALPHA:.2f} %")

pasos, valores = [], []
with open("alpha.dat") as f:
    for linea in f:
        p = linea.split()
        if len(p) >= 2:
            pasos.append(int(p[0])); valores.append(float(p[1].strip("[]")))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(pasos, valores, "o-", color=AZUL, ms=3, label="α estimado")
ax[0].axhline(ALPHA, ls="--", color=ROJO, label="α real = 0,1")
ax[0].set_xlabel("Iteración"); ax[0].set_ylabel("α"); ax[0].set_title("(a) Convergencia del parámetro"); ax[0].legend()
im = ax[1].pcolormesh(XX, TT, modelo_inv.predict(Xg).reshape(XX.shape), cmap="viridis", shading="auto")
fig.colorbar(im, ax=ax[1]); ax[1].scatter(X_obs[:, 0], X_obs[:, 1], s=14, c="white", edgecolors="k", label="mediciones")
ax[1].set_xlabel("x"); ax[1].set_ylabel("t"); ax[1].set_title("(b) Solución reconstruida"); ax[1].legend(loc="upper right")
fig.tight_layout(); plt.show()

## 3. Ecuación de onda 1D: cuerda vibrante

$$u_{tt}=c^2u_{xx},\quad c=1,\qquad u(x,0)=\mathrm{sen}(\pi x)+\tfrac12\,\mathrm{sen}(3\pi x),\quad u_t(x,0)=0,\quad u(0,t)=u(1,t)=0.$$

**Solución analítica.** Por separación de variables, cada modo $\mathrm{sen}(n\pi x)$ oscila con frecuencia $n\pi c$:
$$u(x,t)=\mathrm{sen}(\pi x)\cos(\pi t)+\tfrac12\,\mathrm{sen}(3\pi x)\cos(3\pi t).$$

**PINN con restricción dura:** $\;u_\theta=u(x,0)+x(1-x)\,t^2\,N_\theta(x,t)$ cumple por construcción la condición inicial, $u_t(x,0)=0$ y la frontera; la pérdida se reduce al residuo $R=\partial_{tt}u_\theta-c^2\,\partial_{xx}u_\theta$.

**Arquitectura:** $2-64-64-64-64-1$ · $\tanh$ · Adam $15\,000$ + L-BFGS · 3000 puntos de colocación.

In [ ]:
C_ONDA = 1.0
exacta_onda = lambda X: (np.sin(np.pi * X[:, 0:1]) * np.cos(np.pi * C_ONDA * X[:, 1:2])
                         + 0.5 * np.sin(3 * np.pi * X[:, 0:1]) * np.cos(3 * np.pi * C_ONDA * X[:, 1:2]))

def onda(X, u):
    return dde.grad.hessian(u, X, i=1, j=1) - C_ONDA ** 2 * dde.grad.hessian(u, X, i=0, j=0)

def restriccion(X, y):
    x, t = X[:, 0:1], X[:, 1:2]
    u0 = torch.sin(np.pi * x) + 0.5 * torch.sin(3 * np.pi * x)
    return u0 + x * (1 - x) * t ** 2 * y

dde.config.set_random_seed(1234)
data = dde.data.TimePDE(geom_xt, onda, [], num_domain=3000, num_test=2000, train_distribution="Hammersley")
net = dde.nn.FNN([2] + [64] * 4 + [1], "tanh", "Glorot uniform"); net.apply_output_transform(restriccion)
modelo_onda = dde.Model(data, net)
t0 = time.perf_counter()
modelo_onda.compile("adam", lr=1e-3); modelo_onda.train(iterations=IT(15000), display_every=5000)
modelo_onda.compile("L-BFGS"); modelo_onda.train()
tiempo = time.perf_counter() - t0

ue = exacta_onda(Xg).reshape(XX.shape); up = modelo_onda.predict(Xg).reshape(XX.shape)
print(f"\nOnda 1D: error L2 relativo = {error_l2(up, ue):.2e}   tiempo = {tiempo:.0f} s")
mapas(XX, TT, ue, up)

fig, a = plt.subplots(figsize=(7, 3.8))
for tv, col in [(0, AZUL), (0.25, VERDE), (0.5, ROJO)]:
    i = np.argmin(np.abs(xs - tv))
    a.plot(xs, ue[i], color=col, lw=3, alpha=0.4); a.plot(xs, up[i], "--", color=col, lw=1.8, label=f"t = {tv}")
a.set_xlabel("x"); a.set_ylabel("u(x,t)"); a.set_title("Perfiles de la cuerda: exacta (continua) y PINN (discontinua)")
a.legend(); plt.show()

## 4. Ecuación de onda 2D con velocidad variable

Versión reducida del problema de propagación de ondas sísmicas de Moseley (2022):

$$\Big[\nabla^2-\frac{1}{c(x)^2}\frac{\partial^2}{\partial t^2}\Big]u(x,t)=0,\qquad u(x,0)=e^{-\frac12\|x\|^2/\sigma^2},\qquad \frac{\partial u}{\partial t}(x,0)=0,$$

con $x=(x_1,x_2)\in[-1,1]^2$, $t\in[0;\,0{,}8]$, $\sigma=0{,}15$ y una velocidad que aumenta en una región del medio:
$$c(x)=1+0{,}5\,e^{-\|x-(0{,}5;\,0{,}5)\|^2/(2\cdot0{,}25^2)}.$$

**Interpretación.** $u$ es la perturbación (presión) del medio; la gaussiana inicial es una fuente puntual en reposo (una explosión o el foco de un sismo) y $c(x)$ describe un subsuelo heterogéneo: el frente de onda avanza más rápido donde $c$ es mayor y por eso se deforma.

**Referencia:** no hay solución cerrada; se calcula por diferencias finitas de segundo orden (esquema *leapfrog*) en una malla fina sobre un dominio mayor, $[-2{,}5;\,2{,}5]^2$, para que las reflexiones en la frontera no alcancen la región de estudio.

**PINN con restricción dura:** $\;u_\theta=u(x,0)+(t/T)^2\,N_\theta(x_1,x_2,t)$, que cumple $u(x,0)$ y $u_t(x,0)=0$; la pérdida es el residuo $R=\partial_{tt}u_\theta-c(x)^2(\partial_{x_1x_1}u_\theta+\partial_{x_2x_2}u_\theta)$.

**Arquitectura:** $3-64-64-64-64-1$ · $\tanh$ · Adam $20\,000$ + L-BFGS ($1000$ iteraciones) · 6000 puntos de colocación.

In [ ]:
SIG, T_ONDA = 0.15, 0.8
c_np = lambda x1, x2: 1.0 + 0.5 * np.exp(-((x1 - 0.5) ** 2 + (x2 - 0.5) ** 2) / (2 * 0.25 ** 2))

# --- referencia por diferencias finitas ---
def onda2d_fd(dx=0.01, dt=0.002, L=2.5, tiempos=(0.0, 0.2, 0.4, 0.6, 0.8)):
    x = np.arange(-L, L + dx / 2, dx); X1, X2 = np.meshgrid(x, x, indexing="ij")
    C2 = c_np(X1, X2) ** 2; u0 = np.exp(-0.5 * (X1 ** 2 + X2 ** 2) / SIG ** 2)
    def lap(u):
        l = np.zeros_like(u)
        l[1:-1, 1:-1] = (u[2:, 1:-1] + u[:-2, 1:-1] + u[1:-1, 2:] + u[1:-1, :-2] - 4 * u[1:-1, 1:-1]) / dx ** 2
        return l
    um, u = u0, u0 + 0.5 * dt ** 2 * C2 * lap(u0)          # primer paso con u_t(x,0)=0
    fotos = {0.0: u0.copy()}
    for k in range(2, int(round(max(tiempos) / dt)) + 1):
        um, u = u, 2 * u - um + dt ** 2 * C2 * lap(u)
        for tt in tiempos:
            if abs(k * dt - tt) < 1e-9: fotos[tt] = u.copy()
    m = (x >= -1 - 1e-9) & (x <= 1 + 1e-9)
    return x[m], {tt: v[np.ix_(m, m)] for tt, v in fotos.items()}

t0 = time.perf_counter()
x_fd, ref = onda2d_fd()
print(f"Referencia por diferencias finitas: {time.perf_counter() - t0:.1f} s")

In [ ]:
geom_2d = dde.geometry.GeometryXTime(dde.geometry.Rectangle([-1, -1], [1, 1]), dde.geometry.TimeDomain(0, T_ONDA))

def c2_torch(X):
    x1, x2 = X[:, 0:1], X[:, 1:2]
    return (1.0 + 0.5 * torch.exp(-((x1 - 0.5) ** 2 + (x2 - 0.5) ** 2) / (2 * 0.25 ** 2))) ** 2

def onda2d(X, u):
    u_tt = dde.grad.hessian(u, X, i=2, j=2)
    lap = dde.grad.hessian(u, X, i=0, j=0) + dde.grad.hessian(u, X, i=1, j=1)
    return u_tt - c2_torch(X) * lap

def restriccion_2d(X, y):
    x1, x2, t = X[:, 0:1], X[:, 1:2], X[:, 2:3]
    return torch.exp(-0.5 * (x1 ** 2 + x2 ** 2) / SIG ** 2) + (t / T_ONDA) ** 2 * y

dde.config.set_random_seed(1234)
data = dde.data.TimePDE(geom_2d, onda2d, [], num_domain=6000, num_test=2000, train_distribution="Hammersley")
net = dde.nn.FNN([3] + [64] * 4 + [1], "tanh", "Glorot uniform"); net.apply_output_transform(restriccion_2d)
modelo_2d = dde.Model(data, net)
t0 = time.perf_counter()
modelo_2d.compile("adam", lr=1e-3); modelo_2d.train(iterations=IT(20000), display_every=4000)
dde.optimizers.config.set_LBFGS_options(maxiter=min(1000, LBFGS_MAX))
modelo_2d.compile("L-BFGS"); modelo_2d.train()
dde.optimizers.config.set_LBFGS_options(maxiter=LBFGS_MAX)
print(f"\nTiempo de entrenamiento: {time.perf_counter() - t0:.0f} s")

In [ ]:
X1, X2 = np.meshgrid(x_fd, x_fd, indexing="ij")
tiempos = [0.0, 0.2, 0.4, 0.6, 0.8]
pred = {tt: modelo_2d.predict(np.c_[X1.ravel(), X2.ravel(), np.full(X1.size, tt)]).reshape(X1.shape) for tt in tiempos}
for tt in tiempos[1:]:
    print(f"t = {tt}: error L2 relativo = {error_l2(pred[tt], ref[tt]):.2e}")
print(f"Global (t > 0): {error_l2(np.concatenate([pred[t].ravel() for t in tiempos[1:]]), np.concatenate([ref[t].ravel() for t in tiempos[1:]])):.2e}")

vmax = 0.25
fig, ax = plt.subplots(3, 5, figsize=(17, 10))
filas = [(ref, "Diferencias finitas", "viridis", vmax), (pred, "PINN", "viridis", vmax), (None, "Diferencia", "RdBu_r", 0.05)]
for i, (D, nombre, cm, lim) in enumerate(filas):
    for j, tt in enumerate(tiempos):
        Z = pred[tt] - ref[tt] if D is None else D[tt]
        im = ax[i, j].pcolormesh(X1, X2, Z, cmap=cm, vmin=-lim, vmax=lim, shading="auto")
        ax[i, j].set_aspect("equal"); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        if i == 0: ax[i, j].set_title(f"t = {tt}")
        if j == 0: ax[i, j].set_ylabel(nombre)
    fig.colorbar(im, ax=ax[i, :], shrink=0.85)
plt.show()

## Conclusiones

1. El esquema de las PINNs para EDO se extiende sin cambios conceptuales a EDP: más entradas en la red y derivadas parciales en el residuo, todas por diferenciación automática.
2. En la ecuación del calor y en la cuerda vibrante la PINN alcanza errores pequeños frente a la solución exacta.
3. En el problema inverso del calor, la difusividad $\alpha$ se recupera a partir de 60 mediciones con ruido, en un solo entrenamiento.
4. En la onda 2D con velocidad variable, la PINN reproduce la propagación y la deformación del frente de onda; el error crece con el tiempo, el límite que motiva técnicas como las FBPINN (Moseley et al., 2023), que dividen el dominio en subdominios con redes pequeñas.

---

**Referencias.** Lu, L. et al. (2021). DeepXDE. · Moseley, B. (2022). Tesis doctoral, Universidad de Oxford. · Moseley, B., Markham, A. y Nissen-Meyer, T. (2023). Finite basis physics-informed neural networks (FBPINNs). · Raissi, M., Perdikaris, P. y Karniadakis, G. E. (2019). Physics-informed neural networks.

**Contacto:** Lenin Herrera Padilla · lenin.herrera.p@uni.pe